# 03 — Modélisation

## Objectif
Entraîner un modèle de classification pour prédire la probabilité de défaut de remboursement (`TARGET = 1`).

## Étapes
1. Chargement des données préprocessées
2. Normalisation des features (StandardScaler)
3. Entraînement avec tracking MLflow
4. Évaluation : AUC, matrice de confusion, coût métier

In [1]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

c:\Users\Utilisateur\Desktop\dev\oc\p6\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement des données

In [2]:
df = pd.read_csv('../data/train_preprocessed.csv')
print(f'Shape : {df.shape}')
print(f'\nDistribution TARGET :')
print(df['TARGET'].value_counts(normalize=True).round(3))

# Séparation features / target
y = df['TARGET']
X = df.drop(columns=['TARGET'])

# Nettoyage des noms de colonnes :
# LightGBM rejette les caractères spéciaux JSON ([], {}, :, ", etc.)
# On remplace tout caractère non alphanumérique par un underscore
X.columns = X.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)

# Split stratifié : conserve la proportion de TARGET dans chaque set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nX_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')

Shape : (10000, 227)

Distribution TARGET :
TARGET
0    0.922
1    0.078
Name: proportion, dtype: float64

X_train : (8000, 226)
X_test  : (2000, 226)


## 2. Normalisation (StandardScaler)

La Régression Logistique est sensible à l'échelle des variables.
Sans normalisation, une feature comme `AMT_INCOME_TOTAL` (centaines de milliers)
écraserait un flag 0/1 dans les calculs.

**Règle importante** : on `fit` le scaler **uniquement sur X_train**,
puis on applique la même transformation à X_test.
Fitter sur X_test introduirait du **data leakage** (fuite d'info du futur).

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform sur train
X_test_scaled  = scaler.transform(X_test)        # transform seulement sur test
print('Normalisation OK')

Normalisation OK


## 3. Validation croisée — StratifiedKFold

Pour chaque modèle, on évalue les performances avec **5 folds stratifiés** :
- Chaque fold conserve la même proportion de TARGET=1 (≈8%)
- On obtient une AUC moyenne ± écart-type, plus fiable qu'un seul split

**Fonction `cv_evaluate`** : prend un modèle, X_train, y_train et retourne les métriques moyennes sur 5 folds.
Le paramètre `scale=True` est uniquement pour la Régression Logistique (sensible à l'échelle) : le scaler est fitté à l'intérieur de chaque fold pour éviter tout data leakage.

In [4]:
def cv_evaluate(model, X, y, n_splits=5, scale=False):
    """
    Évalue un modèle par validation croisée StratifiedKFold.

    Paramètres :
    - model    : modèle scikit-learn compatible (fit / predict_proba)
    - X, y     : features et target (sur X_train uniquement)
    - n_splits : nombre de folds (5 par défaut)
    - scale    : True pour appliquer StandardScaler à l'intérieur de chaque fold
                 (nécessaire pour LR, inutile pour les arbres)

    Retourne un dict avec auc_mean, auc_std, cost_mean, fn_mean, fp_mean.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs, costs, fns, fps = [], [], [], []

    for train_idx, val_idx in skf.split(X, y):
        X_tr  = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        if scale:
            sc    = StandardScaler()
            X_tr  = sc.fit_transform(X_tr)   # fit sur le fold train uniquement
            X_val = sc.transform(X_val)       # transform sur le fold val

        model.fit(X_tr, y_tr)
        y_proba = model.predict_proba(X_val)[:, 1]
        y_pred  = (y_proba >= 0.5).astype(int)

        auc = roc_auc_score(y_val, y_proba)
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

        aucs.append(auc)
        costs.append(10 * fn + fp)
        fns.append(fn)
        fps.append(fp)

    return {
        'auc_mean':  np.mean(aucs),
        'auc_std':   np.std(aucs),
        'cost_mean': np.mean(costs),
        'fn_mean':   np.mean(fns),
        'fp_mean':   np.mean(fps),
    }

print('Fonction cv_evaluate prête')

Fonction cv_evaluate prête


## 3. Entraînement — Régression Logistique (baseline)

Choix des paramètres :
- `class_weight='balanced'` : compense le déséquilibre des classes (beaucoup plus de 0 que de 1)
- `max_iter=2000` : assez d'itérations pour que l'algorithme converge
- `solver='lbfgs'` : algorithme d'optimisation adapté aux datasets de taille moyenne

In [5]:
mlflow.set_experiment('scoring_credit')

params = {
    'solver': 'lbfgs',
    'max_iter': 2000,
    'random_state': 42,
    'class_weight': 'balanced',
}

with mlflow.start_run(run_name='LogisticRegression_baseline'):

    mlflow.set_tag('model', 'LogisticRegression')

    # --- Validation croisée (5 folds, avec scaling dans chaque fold) ---
    lr = LogisticRegression(**params)
    cv = cv_evaluate(lr, X_train, y_train, scale=True)
    print(f"CV AUC  : {cv['auc_mean']:.4f} ± {cv['auc_std']:.4f}")
    print(f"CV Coût : {cv['cost_mean']:.1f}  (FN≈{cv['fn_mean']:.1f}, FP≈{cv['fp_mean']:.1f})")

    # --- Modèle final entraîné sur tout X_train, évalué sur X_test (holdout) ---
    lr_final = LogisticRegression(**params)
    lr_final.fit(X_train_scaled, y_train)
    y_pred  = lr_final.predict(X_test_scaled)
    y_proba = lr_final.predict_proba(X_test_scaled)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp

    # --- Log MLflow ---
    mlflow.log_params(params)
    mlflow.log_metric('cv_auc_mean',  cv['auc_mean'])
    mlflow.log_metric('cv_auc_std',   cv['auc_std'])
    mlflow.log_metric('test_auc',     auc)
    mlflow.log_metric('f1',           f1)
    mlflow.log_metric('cout_metier',  cout_metier)
    mlflow.log_metric('fn',           fn)
    mlflow.log_metric('fp',           fp)
    mlflow.sklearn.log_model(lr_final, name='model', registered_model_name='ScoringCredit_LR')

    # --- Affichage holdout ---
    print(f"\nHoldout AUC : {auc:.4f}  (cible : 0.75 - 0.82)")
    print(f"F1          : {f1:.4f}")
    print(f"Coût métier : {cout_metier}  (FN={fn}, FP={fp})")
    print(f'\n{classification_report(y_test, y_pred)}')

CV AUC  : 0.7041 ± 0.0213
CV Coût : 944.4  (FN≈52.4, FP≈420.4)


2026/03/02 23:52:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Holdout AUC : 0.6956  (cible : 0.75 - 0.82)
F1          : 0.2249
Coût métier : 1201  (FN=63, FP=571)

              precision    recall  f1-score   support

           0       0.95      0.69      0.80      1845
           1       0.14      0.59      0.22       155

    accuracy                           0.68      2000
   macro avg       0.55      0.64      0.51      2000
weighted avg       0.89      0.68      0.76      2000



Successfully registered model 'ScoringCredit_LR'.
Created version '1' of model 'ScoringCredit_LR'.


## 4. Entraînement — Random Forest

Le Random Forest est un modèle à base d'**arbres de décision** :
- Il n'est **pas sensible à l'échelle** des variables → pas besoin de StandardScaler
- `n_estimators=100` : 100 arbres construits en parallèle, le vote majoritaire donne la prédiction finale
- `max_depth=10` : limite la profondeur de chaque arbre pour éviter l'overfitting
- `n_jobs=-1` : utilise tous les CPU disponibles pour accélérer l'entraînement

In [6]:
params_rf = {
    'n_estimators': 100,
    'max_depth': 10,
    'random_state': 42,
    'class_weight': 'balanced',
    'n_jobs': -1,
}

with mlflow.start_run(run_name='RandomForest_baseline'):

    mlflow.set_tag('model', 'RandomForest')

    # --- Validation croisée (pas de scaling pour les arbres) ---
    rf = RandomForestClassifier(**params_rf)
    cv = cv_evaluate(rf, X_train, y_train, scale=False)
    print(f"CV AUC  : {cv['auc_mean']:.4f} ± {cv['auc_std']:.4f}")
    print(f"CV Coût : {cv['cost_mean']:.1f}  (FN≈{cv['fn_mean']:.1f}, FP≈{cv['fp_mean']:.1f})")

    # --- Modèle final entraîné sur tout X_train, évalué sur X_test (holdout) ---
    rf_final = RandomForestClassifier(**params_rf)
    rf_final.fit(X_train, y_train)
    y_pred  = rf_final.predict(X_test)
    y_proba = rf_final.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp

    # --- Log MLflow ---
    mlflow.log_params(params_rf)
    mlflow.log_metric('cv_auc_mean',  cv['auc_mean'])
    mlflow.log_metric('cv_auc_std',   cv['auc_std'])
    mlflow.log_metric('test_auc',     auc)
    mlflow.log_metric('f1',           f1)
    mlflow.log_metric('cout_metier',  cout_metier)
    mlflow.log_metric('fn',           fn)
    mlflow.log_metric('fp',           fp)
    mlflow.sklearn.log_model(rf_final, name='model', registered_model_name='ScoringCredit_RF')

    # --- Affichage holdout ---
    print(f"\nHoldout AUC : {auc:.4f}  (cible : 0.75 - 0.82)")
    print(f"F1          : {f1:.4f}")
    print(f"Coût métier : {cout_metier}  (FN={fn}, FP={fp})")
    print(f'\n{classification_report(y_test, y_pred)}')

CV AUC  : 0.7018 ± 0.0121
CV Coût : 1037.4  (FN≈94.6, FP≈91.4)


2026/03/02 23:52:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Holdout AUC : 0.7083  (cible : 0.75 - 0.82)
F1          : 0.2553
Coût métier : 1262  (FN=113, FP=132)

              precision    recall  f1-score   support

           0       0.94      0.93      0.93      1845
           1       0.24      0.27      0.26       155

    accuracy                           0.88      2000
   macro avg       0.59      0.60      0.59      2000
weighted avg       0.88      0.88      0.88      2000



Successfully registered model 'ScoringCredit_RF'.
Created version '1' of model 'ScoringCredit_RF'.


## 5. Entraînement — XGBoost

XGBoost est un algorithme de **gradient boosting** : il construit les arbres **séquentiellement**,
chaque arbre corrigeant les erreurs du précédent (contrairement au RF qui les construit en parallèle).

Choix des paramètres :
- `scale_pos_weight` : remplace `class_weight='balanced'` pour XGBoost. Il vaut `nb_négatifs / nb_positifs` pour compenser le déséquilibre
- `n_estimators=200` : 200 arbres en séquence
- `max_depth=6` : profondeur standard pour le boosting (moins profond que RF car les arbres se corrigent entre eux)
- `learning_rate=0.1` : taille du pas d'apprentissage — plus petit = plus précis mais plus lent
- `use_label_encoder=False` + `eval_metric='logloss'` : suppriment des warnings XGBoost

In [7]:
# Calcul du scale_pos_weight : ratio négatifs / positifs
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight : {scale_pos_weight:.2f}  ({neg} négatifs / {pos} positifs)')

params_xgb = {
    'n_estimators':     200,
    'max_depth':        6,
    'learning_rate':    0.1,
    'scale_pos_weight': scale_pos_weight,
    'random_state':     42,
    'eval_metric':      'logloss',
    'n_jobs':           -1,
}

with mlflow.start_run(run_name='XGBoost_baseline'):

    mlflow.set_tag('model', 'XGBoost')

    # --- Validation croisée ---
    xgb = XGBClassifier(**params_xgb)
    cv = cv_evaluate(xgb, X_train, y_train, scale=False)
    print(f"CV AUC  : {cv['auc_mean']:.4f} ± {cv['auc_std']:.4f}")
    print(f"CV Coût : {cv['cost_mean']:.1f}  (FN≈{cv['fn_mean']:.1f}, FP≈{cv['fp_mean']:.1f})")

    # --- Modèle final entraîné sur tout X_train, évalué sur X_test (holdout) ---
    xgb_final = XGBClassifier(**params_xgb)
    xgb_final.fit(X_train, y_train)
    y_pred  = xgb_final.predict(X_test)
    y_proba = xgb_final.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp

    # --- Log MLflow ---
    mlflow.log_params(params_xgb)
    mlflow.log_metric('cv_auc_mean',  cv['auc_mean'])
    mlflow.log_metric('cv_auc_std',   cv['auc_std'])
    mlflow.log_metric('test_auc',     auc)
    mlflow.log_metric('f1',           f1)
    mlflow.log_metric('cout_metier',  cout_metier)
    mlflow.log_metric('fn',           fn)
    mlflow.log_metric('fp',           fp)
    mlflow.sklearn.log_model(xgb_final, name='model', registered_model_name='ScoringCredit_XGBoost')

    # --- Affichage holdout ---
    print(f"\nHoldout AUC : {auc:.4f}  (cible : 0.75 - 0.82)")
    print(f"F1          : {f1:.4f}")
    print(f"Coût métier : {cout_metier}  (FN={fn}, FP={fp})")
    print(f'\n{classification_report(y_test, y_pred)}')

scale_pos_weight : 11.90  (7380 négatifs / 620 positifs)
CV AUC  : 0.6991 ± 0.0194
CV Coût : 1057.0  (FN≈98.6, FP≈71.0)


2026/03/02 23:53:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Holdout AUC : 0.7433  (cible : 0.75 - 0.82)
F1          : 0.2589
Coût métier : 1264  (FN=115, FP=114)

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1845
           1       0.26      0.26      0.26       155

    accuracy                           0.89      2000
   macro avg       0.60      0.60      0.60      2000
weighted avg       0.89      0.89      0.89      2000



Successfully registered model 'ScoringCredit_XGBoost'.
Created version '1' of model 'ScoringCredit_XGBoost'.


## 6. Entraînement — LightGBM

LightGBM est aussi un **gradient boosting**, concurrent direct de XGBoost, avec deux avantages :
- **Plus rapide** : utilise un algorithme de découpage des feuilles différent (*leaf-wise* vs *level-wise*)
- **Moins gourmand en mémoire** : travaille sur des histogrammes de valeurs plutôt que sur les valeurs brutes

Paramètres :
- `scale_pos_weight` : même logique que XGBoost pour gérer le déséquilibre
- `n_estimators=200` : même nombre d'arbres que XGBoost pour une comparaison équitable
- `max_depth=6` : même profondeur
- `learning_rate=0.1` : même taux d'apprentissage
- `verbose=-1` : supprime les logs verbeux de LightGBM

In [8]:
params_lgbm = {
    'n_estimators':     200,
    'max_depth':        6,
    'learning_rate':    0.1,
    'scale_pos_weight': scale_pos_weight,  # calculé dans la cellule XGBoost
    'random_state':     42,
    'verbose':          -1,
    'n_jobs':           -1,
}

with mlflow.start_run(run_name='LightGBM_baseline'):

    mlflow.set_tag('model', 'LightGBM')

    # --- Validation croisée ---
    lgbm = LGBMClassifier(**params_lgbm)
    cv = cv_evaluate(lgbm, X_train, y_train, scale=False)
    print(f"CV AUC  : {cv['auc_mean']:.4f} ± {cv['auc_std']:.4f}")
    print(f"CV Coût : {cv['cost_mean']:.1f}  (FN≈{cv['fn_mean']:.1f}, FP≈{cv['fp_mean']:.1f})")

    # --- Modèle final entraîné sur tout X_train, évalué sur X_test (holdout) ---
    lgbm_final = LGBMClassifier(**params_lgbm)
    lgbm_final.fit(X_train, y_train)
    y_pred  = lgbm_final.predict(X_test)
    y_proba = lgbm_final.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cout_metier = 10 * fn + 1 * fp

    # --- Log MLflow ---
    mlflow.log_params(params_lgbm)
    mlflow.log_metric('cv_auc_mean',  cv['auc_mean'])
    mlflow.log_metric('cv_auc_std',   cv['auc_std'])
    mlflow.log_metric('test_auc',     auc)
    mlflow.log_metric('f1',           f1)
    mlflow.log_metric('cout_metier',  cout_metier)
    mlflow.log_metric('fn',           fn)
    mlflow.log_metric('fp',           fp)
    mlflow.sklearn.log_model(lgbm_final, name='model', registered_model_name='ScoringCredit_LightGBM')

    # --- Affichage holdout ---
    print(f"\nHoldout AUC : {auc:.4f}  (cible : 0.75 - 0.82)")
    print(f"F1          : {f1:.4f}")
    print(f"Coût métier : {cout_metier}  (FN={fn}, FP={fp})")
    print(f'\n{classification_report(y_test, y_pred)}')

CV AUC  : 0.7004 ± 0.0183
CV Coût : 1070.2  (FN≈99.2, FP≈78.2)


2026/03/02 23:53:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Holdout AUC : 0.7258  (cible : 0.75 - 0.82)
F1          : 0.2866
Coût métier : 1210  (FN=109, FP=120)

              precision    recall  f1-score   support

           0       0.94      0.93      0.94      1845
           1       0.28      0.30      0.29       155

    accuracy                           0.89      2000
   macro avg       0.61      0.62      0.61      2000
weighted avg       0.89      0.89      0.89      2000



Successfully registered model 'ScoringCredit_LightGBM'.
Created version '1' of model 'ScoringCredit_LightGBM'.
